In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score
import xarray as xr
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import ensemble

In [2]:
scores = xr.open_dataset("./scores/scores_final.nc")

param0 = pd.read_csv('../proccessing_ensemble/lhs_np15_ns3000_values.txt', sep='\s+')
ds = param0.to_xarray()
param0 = ds.rename({'index': 'sim'})

param=param0.drop_vars(['ytill.cf_negis_centre', 'ytill.cf_negis_south'])
new_order = ['itm.itm_b', 'itm.itm_c', 'marine_shelf.kappa_grz','snap.f_p_ne','ymat.enh_shear','ydyn.beta_q', 'ytill.z0','yneff.delta','ytopo.kt','isostasy.tau','isostasy.He_lith','ytill.cf_negis_north','ctrl.cf_negis_1']
param = param[new_order]

y = scores["S"].to_dataframe()
X = param.to_dataframe()

train_ds, val_ds = train_test_split(scores.sim.values, train_size=0.9, test_size=0.1)
y = scores["S"].to_dataframe()
X = param.to_dataframe()

df_X_train = X.loc[train_ds]
df_X_val = X.loc[val_ds]
df_y_train = y.loc[train_ds]
df_y_val = y.loc[val_ds]

X_train = df_X_train.values
X_val = df_X_val.values
y_train = df_y_train.values
y_val = df_y_val.values




<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3943949/2670593228.py:3: SyntaxWarning: invalid escape sequence '\s'
  param0 = pd.read_csv('../proccessing_ensemble/lhs_np15_ns3000_values.txt', sep='\s+')


In [3]:
param_names = {
    'itm.itm_b': 'b',
    'itm.itm_c': 'c$_{\mathrm{1}}$',
    'ydyn.beta_q': 'q',
    'ytill.z0': 'z$_{\mathrm{0}}$',
    'yneff.delta': '$\delta$',
    'ymat.enh_shear': 'E$_{\mathrm{shear}}$',
    'marine_shelf.kappa_grz': '$\kappa$',
    'isostasy.tau': '$\\tau$',
    'isostasy.He_lith': 'H$_{\mathrm{e}}$',
    'ytopo.kt': 'K$_{\mathrm{t}}$',
    'ytill.cf_negis_north': 'f$_{\mathrm{mid}}$',
    'snap.f_p_ne': 'f$_{\mathrm{p}}$',
    'ctrl.cf_negis_1': 'f$_{\mathrm{low}}$'
}

param_units = {
    'itm.itm_b': 'W m$^{-2}$',
    'itm.itm_c': 'W m$^{-2}$',
    'ydyn.beta_q': '-',
    'ytill.z0': 'm',
    'yneff.delta': '-',
    'ymat.enh_shear': '-',
    'marine_shelf.kappa_grz': 'm yr$^{-1}$ K$^{-1}$',
    'isostasy.tau': 'yr',
    'isostasy.He_lith': 'km',
    'ytopo.kt': 'm yr$^{-1}$ Pa$^{-1}$',
    'ytill.cf_negis_north': '-',
    'snap.f_p_ne': '-',
    'ctrl.cf_negis_1': '-'
}
scores_names={"S": "S",
    "S_H_ice": "S$_{\mathrm{Hice}}$",
    "S_ice_cover": "S$_{{\mathrm{ice \, cover}}}$",
    "S_z_bed": "S$_{\mathrm{zbed}}$",
    "S_uxy": "S$_{\mathrm{uxy}}$",
    "S_grip": "S$_{\mathrm{GRIP}}$",
    "S_ngrip": "S$_{\mathrm{NGRIP}}$",
    "S_dye3": "S$_{\mathrm{DYE3}}$",
    "S_isocr": "S$_{\mathrm{iso}}$",
    "S_lgm": "S$_{\mathrm{LGM}}$"
}

feature_names = [param_names.get(col, col) for col in X.columns]
feature_unit = [param_units.get(col, col) for col in X.columns]

<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:5: SyntaxWarning: invalid escape sequence '\m'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\k'
<>:10: SyntaxWarning: invalid escape sequence '\m'
<>:11: SyntaxWarning: invalid escape sequence '\m'
<>:12: SyntaxWarning: invalid escape sequence '\m'
<>:13: SyntaxWarning: invalid escape sequence '\m'
<>:14: SyntaxWarning: invalid escape sequence '\m'
<>:33: SyntaxWarning: invalid escape sequence '\m'
<>:34: SyntaxWarning: invalid escape sequence '\m'
<>:35: SyntaxWarning: invalid escape sequence '\m'
<>:36: SyntaxWarning: invalid escape sequence '\m'
<>:37: SyntaxWarning: invalid escape sequence '\m'
<>:38: SyntaxWarning: invalid escape sequence '\m'
<>:39: SyntaxWarning: invalid escape sequence '\m'
<>:40: SyntaxWarning: invalid escape sequence '\m'
<>:41: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape 

In [4]:
# Best model with xgb.XGBRegressor

param_grid = {
    'n_estimators': [500, 1000],           # Más árboles para captar patrones complejos
    'learning_rate': [0.01, 0.05, 0.1],    # Añadimos 0.1 por si necesita aprender más rápido
    'max_depth': [3, 4, 5],                # Más profundidad (el anterior de 3 era muy poco)
    'subsample': [0.8, 0.9],               # Usar más datos por árbol
    'colsample_bytree': [0.7, 0.8],        # Proporción de variables por árbol
    'reg_lambda': [1, 10, 50, 100],                 # Regularización L2 para evitar el overfitting del ensemble
    'reg_alpha': [0, 0.1]                  # Regularización L1
}

grid_search = GridSearchCV(
    estimator=xgb.XGBRegressor(
        random_state=42, 
        objective='reg:squarederror',
        tree_method='hist'
    ),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor R2 en Validación Cruzada: {grid_search.best_score_:.4f}")
#  11 mins

Fitting 5 folds for each of 576 candidates, totalling 2880 fits


Mejores parámetros: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 1000, 'reg_alpha': 0, 'reg_lambda': 10, 'subsample': 0.8}
Mejor R2 en Validación Cruzada: 0.3352


In [5]:

train_r2 = best_model.score(X_train, y_train)
test_r2 = best_model.score(X_val, y_val)

print(f"R2 Final Entrenamiento: {train_r2:.4f}")
print(f"R2 Final Test (el que cuenta): {test_r2:.4f}")

best_model.save_model("./output/xgboost_for_scores_final.json")

R2 Final Entrenamiento: 0.6044
R2 Final Test (el que cuenta): 0.4257


In [8]:
# Now we repeat por S_NGRIP

y = scores["S_ngrip"].to_dataframe()
X = param.to_dataframe()

train_ds, val_ds = train_test_split(scores.sim.values, train_size=0.9, test_size=0.1)
y = scores["S_ngrip"].to_dataframe()
X = param.to_dataframe()

df_X_train = X.loc[train_ds]
df_X_val = X.loc[val_ds]
df_y_train = y.loc[train_ds]
df_y_val = y.loc[val_ds]

X_train = df_X_train.values
X_val = df_X_val.values
y_train = df_y_train.values
y_val = df_y_val.values

# Best model with xgb.XGBRegressor

param_grid = {
    'n_estimators': [500, 1000],           # Más árboles para captar patrones complejos
    'learning_rate': [0.01, 0.05, 0.1],    # Añadimos 0.1 por si necesita aprender más rápido
    'max_depth': [3, 4],             
    'subsample': [0.8, 0.9],               # Usar más datos por árbol
    'colsample_bytree': [0.7, 0.8],        # Proporción de variables por árbol
    'reg_lambda': [10, 50, 100],          # Regularización L2 para evitar el overfitting del ensemble
    'reg_alpha': [0, 0.1]                  # Regularización L1
}

grid_search = GridSearchCV(
    estimator=xgb.XGBRegressor(
        random_state=42, 
        objective='reg:squarederror',
        tree_method='hist'
    ),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1)

grid_search.fit(X_train, y_train)

best_model_ngrip = grid_search.best_estimator_

print("Modelo para S_NGRIP")
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor R2 en Validación Cruzada: {grid_search.best_score_:.4f}")
#  11 mins


Fitting 5 folds for each of 288 candidates, totalling 1440 fits


Modelo para S_NGRIP
Mejores parámetros: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 1000, 'reg_alpha': 0, 'reg_lambda': 100, 'subsample': 0.8}
Mejor R2 en Validación Cruzada: 0.5278


In [9]:

train_r2 = best_model_ngrip.score(X_train, y_train)
test_r2 = best_model_ngrip.score(X_val, y_val)

print(f"R2 Final Entrenamiento: {train_r2:.4f}")
print(f"R2 Final Test (el que cuenta): {test_r2:.4f}")

best_model_ngrip.save_model("./output/xgboost_for_scores_ngrip_final.json")

R2 Final Entrenamiento: 0.8650
R2 Final Test (el que cuenta): 0.5064
